In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.manifold import TSNE

DISTINCT_COLORS = [
    "#E6194B",  
    "#3CB44B", 
    "#4363D8", 
    "#F58231",  
    "#911EB4",  
]
SEED = 42

In [ ]:
original_data   = np.load('../duomenys/original_data.npy')
labels          = np.load('../duomenys/ship-types.npy')
distance_matrix = np.load('../duomenys/dist_mat.npy')

In [ ]:
unique_labels, labels_encoded = np.unique(labels, return_inverse=True)
label_names = {i: name for i, name in enumerate(unique_labels)}

n_samples = original_data.shape[0]
original_flat = original_data.reshape(n_samples, -1)

all_classes = np.unique(labels_encoded)
class_colors = {cls: DISTINCT_COLORS[i % len(DISTINCT_COLORS)]
                for i, cls in enumerate(all_classes)}

embedding = TSNE(
    perplexity=5,
    early_exaggeration=24,
    learning_rate=10,
    max_iter=500,
    n_components=2,
    random_state=SEED
).fit_transform(distance_matrix)

In [ ]:
from sklearn.model_selection import train_test_split

original_flat = original_data.reshape((-1, original_data.shape[1]*original_data.shape[2]))
X_train, X_test, y_train, y_test, emb_train, emb_test = train_test_split(
    original_flat, labels_encoded, embedding, test_size=0.3, random_state=SEED, stratify=labels_encoded
)

In [ ]:
import pandas as pd

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
def run_xgboost(X_train, X_test, y_train, y_test, label_names, **params):

    from sklearn.model_selection import StratifiedKFold, cross_validate

    xgb = XGBClassifier(eval_metric='mlogloss', **params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
    }
    cv_scores = cross_validate(
        xgb, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )

    print(f"\n{'='*60}")
    for param, v in params.items():
        print(f"{param} : {v}")
    print(f"{'='*60}")
    print("Klasifikavimo lentele (true/pred)")

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm)
    print(cm_df.to_string())

    n       = cm.sum()
    diag    = np.diag(cm)
    rowsums = cm.sum(axis=1)
    colsums = cm.sum(axis=0)

    precision = np.where(colsums > 0, diag / colsums, 0.0)
    recall    = np.where(rowsums > 0, diag / rowsums, 0.0)
    f1        = np.where((precision + recall) > 0,
                         2 * precision * recall / (precision + recall), 0.0)
    accuracy  = diag.sum() / n

    print("\nMetrikos kiekvienai klasei:")
    print(pd.DataFrame({"precision": precision, "recall": recall, "f1": f1},
                       index=[label_names[c] for c in range(5)]).round(4).to_string())
    print(f"\nAccuracy      : {accuracy:.4f}")
    print(f"Macro precision : {precision.mean():.4f}")
    print(f"Macro recall    : {recall.mean():.4f}")
    print(f"Macro F1        : {f1.mean():.4f}")

    print("\nCross-validation (macro) metrics:")
    for key in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]:
        values = cv_scores[f"test_{key}"]
        print(f"{key:16s}: {values.mean():.4f} ± {values.std():.4f}")

    return xgb

In [ ]:
xgb_orig = run_xgboost(X_train, X_test, y_train, y_test, label_names)

In [ ]:
X_train_tsne, X_test_tsne, y_train_tsne, y_test_tsne = train_test_split(
    embedding, labels_encoded, test_size=0.3, random_state=SEED, stratify=labels_encoded
)

In [ ]:
xgb_proj = run_xgboost(X_train_tsne, X_test_tsne, y_train_tsne, y_test_tsne, label_names)

In [ ]:
import matplotlib.colors as mcolors

In [ ]:
def plot_decision_boundary(
    clf, 
    X_test_sc,
    y_test,
    label_names,
    title,
    ax,
    unique_classes,
    step=0.3
):

    x_min = X_test_sc[:,0].min() - 1
    x_max = X_test_sc[:,0].max() + 1
    y_min = X_test_sc[:,1].min() - 1
    y_max = X_test_sc[:,1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, step),
                         np.arange(y_min, y_max, step))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])

    class_to_idx = {c: i for i, c in enumerate(unique_classes)}
    Z_idx = np.array([class_to_idx[z] for z in Z]).reshape(xx.shape)
    n_classes = len(unique_classes)
    bg_cmap = mcolors.ListedColormap([class_colors[c] for c in unique_classes])
    ax.contourf(xx, yy, Z_idx, alpha=0.25, cmap=bg_cmap,
                levels=np.arange(-0.5, n_classes, 1))

    # Mokymo taškai
    for cls in unique_classes:
        mask = y_test == cls
        ax.scatter(X_test_sc[mask, 0], X_test_sc[mask, 1],
                   c=class_colors[cls], edgecolors='k', s=60,
                   label=label_names[cls], zorder=3)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(fontsize=7, loc="best")




In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()

plot_decision_boundary(
    xgb_proj, X_test_tsne, y_test_tsne, label_names, unique_classes=all_classes,
    title="XGBoost klasifikavimas t-SNE erdvėje (Testavimo aibė)",
    ax=ax
)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8, 6))

unique_cls = np.unique(labels_encoded)
y_pred = xgb_orig.predict(X_test)
correct = y_pred == y_test 

# Teisingai klasifikuoti — permatomi (alpha=0.4)
ax2.scatter(emb_test[correct, 0], emb_test[correct, 1],
            c=[class_colors[c] for c in y_test[correct]],
            s=50, edgecolors='k', linewidths=0.5, marker='o', alpha=0.4, zorder=2)

for cls in unique_cls:
    mask = correct & (y_test == cls)
    ax2.scatter(
        emb_test[mask, 0], emb_test[mask, 1],
        c=class_colors[cls], s=50, edgecolors='k', linewidths=0.5,
        marker='o', alpha=0.4, zorder=2,
        label=unique_labels[cls]
    )

# Klaidingai klasifikuoti — ryškūs ir didesni (alpha=1.0, s=120)
ax2.scatter(
    emb_test[~correct, 0], emb_test[~correct, 1],
    c=[class_colors[c] for c in y_test[~correct]],
    s=120, edgecolors='red', linewidths=0.5, marker='X', alpha=1.0, zorder=3
)
ax2.legend(fontsize=8, loc='upper right', title="Laivo tipas / rezultatas")

ax2.set_title("XGBoost klaidos", fontsize=11)
ax2.set_xlabel("t-SNE dim 1")
ax2.set_ylabel("t-SNE dim 2")
plt.tight_layout()
# plt.savefig('./plots/knn_errors_orig.svg', format='svg')
plt.show()


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import pandas as pd

param_grid = {
    'n_estimators': [20, 75, 100, 200],
    'max_depth':    [3, 5, 8, 12],
    'learning_rate': [0.05, 0.1, 0.2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='mlogloss', random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

# ── Results ───────────────────────────────────────────────────────────────────
print(f"Best params:  {grid_search.best_params_}")
print(f"Best CV acc:  {grid_search.best_score_:.4f}")

# Full results as a sorted DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)
cols = ['param_n_estimators', 'param_max_depth', 'param_learning_rate',
        'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
print(results_df[cols].sort_values('rank_test_score').to_string(index=False))

# ── Best model ready to use ───────────────────────────────────────────────────
xgb_best = grid_search.best_estimator_

In [ ]:
xgb_optim = run_xgboost(X_train, X_test, y_train, y_test, label_names,
                            **grid_search.best_params_)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8, 6))

unique_cls = np.unique(labels_encoded)
y_pred = xgb_optim.predict(X_test)
correct = y_pred == y_test 

# Teisingai klasifikuoti — permatomi (alpha=0.4)
ax2.scatter(emb_test[correct, 0], emb_test[correct, 1],
            c=[class_colors[c] for c in y_test[correct]],
            s=50, edgecolors='k', linewidths=0.5, marker='o', alpha=0.4, zorder=2)

for cls in unique_cls:
    mask = correct & (y_test == cls)
    ax2.scatter(
        emb_test[mask, 0], emb_test[mask, 1],
        c=class_colors[cls], s=50, edgecolors='k', linewidths=0.5,
        marker='o', alpha=0.4, zorder=2,
        label=unique_labels[cls]
    )

# Klaidingai klasifikuoti — ryškūs ir didesni (alpha=1.0, s=120)
ax2.scatter(
    emb_test[~correct, 0], emb_test[~correct, 1],
    c=[class_colors[c] for c in y_test[~correct]],
    s=120, edgecolors='red', linewidths=0.5, marker='X', alpha=1.0, zorder=3
)
ax2.legend(fontsize=8, loc='upper right', title="Laivo tipas / rezultatas")

ax2.set_title("Optimalaus XGBoost klaidos", fontsize=11)
ax2.set_xlabel("t-SNE dim 1")
ax2.set_ylabel("t-SNE dim 2")
plt.tight_layout()
# plt.savefig('./plots/knn_errors_orig.svg', format='svg')
plt.show()


## Požymių svarba

In [ ]:
n_timesteps = original_data.shape[1]  # 25
n_features  = original_data.shape[2]  # 10
feature_names_short = [
    'Latitude', 'Longitude', 'ROT',
    'SOG', 'Heading_sin', 'Heading_cos',
    'COG_sin', 'COG_cos', 'delta_lat', 'delta_lon'
]

# (250,) -> (25, 10)
imp_2d = xgb_optim.feature_importances_.reshape(n_timesteps, n_features)
importances_agg = imp_2d.mean(axis=0)

xgb_importances = pd.Series(importances_agg, index=feature_names_short)

fig, ax = plt.subplots(figsize=(8, 5))
xgb_importances.plot.bar(yerr=0, ax=ax)
ax.set_ylabel("MDI")
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()

SOG and Latitude are the most important features for distinguishing ship types. Longitude is moderately important. ROT, Heading, COG, delta_lat, and delta_lon contribute relatively little.

The practical takeaway is the same as in the RF notebook: ship type is driven primarily by speed and geography, not by heading dynamics.

In [ ]:
from sklearn.feature_selection import RFECV

xgb_rfe_cv = RFECV(
    estimator=XGBClassifier(
        eval_metric='mlogloss',
        random_state=100,
        n_jobs=-1,
        **grid_search.best_params_,
    ),
    min_features_to_select=50,
    step=25,
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=100),
)

xgb_rfe_cv.fit(
    X_train, y_train
)

In [ ]:
selected_mask = xgb_rfe_cv.support_
selected_features = np.array([
    f'{feat}-{step}'
    for step in range(25)
    for feat in feature_names_short
])[selected_mask]
selected_features

fig, ax = plt.subplots()

ax.imshow(
    xgb_rfe_cv.support_.reshape(25, 10).T,
    cmap='binary', aspect='auto'
)

ax.set_ylabel("Požymis")
ax.set_xlabel("Laiko tarpas")
ax.set_yticks(ticks=range(10), labels=feature_names_short)
ax.set_xticks(range(25))

ax.set_title('Reikšmingi XGBoost kintamieji')

plt.show()

In [ ]:
xgb_final = XGBClassifier(
    eval_metric='mlogloss',
    random_state=100,
    n_jobs=-1,
    **grid_search.best_params_,
)
xgb_final.fit(X_train[:, selected_mask], y_train)

y_pred = xgb_final.predict(original_flat[:, selected_mask])
correct_all = y_pred == labels_encoded

In [ ]:
mmsis = np.load('../duomenys/mmsi-sarasas.npy', allow_pickle=True)

print(f"MMSIs per trajectory check: {mmsis.shape}")  # should be (750,)

print("\nMisclassified MMSIs:")

MISCLASSIFIED_PREDS = {}

for i in np.where(~correct_all)[0]:
    true_label = label_names[labels_encoded[i]]
    pred_label = label_names[y_pred[i]]
    mmsi = mmsis[i]
    print(f"  MMSI {mmsi}  |  true: {true_label:20s}  |  predicted: {pred_label}")

    if mmsi not in MISCLASSIFIED_PREDS:
        MISCLASSIFIED_PREDS[mmsi] = [pred_label]
    else:
        MISCLASSIFIED_PREDS[mmsi].append(pred_label)

## Palyginimas

In [ ]:
from sklearn.model_selection import cross_validate

best_param = grid_search.best_params_

xgb_orig = XGBClassifier(eval_metric='mlogloss', random_state=42, **best_param)
acc = cross_validate(
    xgb_orig,
    original_flat, labels_encoded,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
)

print(f"Tikslumas originaliu: {acc['test_score'].mean()*100:.2f} ± {acc['test_score'].std()*100:.2f}")

xgb_proj = XGBClassifier(eval_metric='mlogloss', random_state=42, **best_param)
acc = cross_validate(
    xgb_proj,
    embedding, labels_encoded,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
)

print(f"Tikslumas projektuotu: {acc['test_score'].mean()*100:.2f} ± {acc['test_score'].std()*100:.2f}")

xgb_sel = XGBClassifier(eval_metric='mlogloss', random_state=42, **best_param)
acc = cross_validate(
    xgb_sel,
    original_flat[:, selected_mask], labels_encoded,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
)

print(f"Tikslumas atrinktu: {acc['test_score'].mean()*100:.2f} ± {acc['test_score'].std()*100:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(imp_2d, aspect='auto', cmap='YlOrRd')

ax.set_xlabel("Požymis")
ax.set_ylabel("Laiko žingsnis")
ax.set_title("Požymių svarbos tankio žemėlapis")

ax.set_xticks(range(n_features))
ax.set_xticklabels(feature_names_short, rotation=45, ha="right")

ax.set_yticks(range(n_timesteps))

plt.colorbar(im, ax=ax, label="Svarba")

fig.tight_layout()
plt.show()